> VIGOR 2.12.1 example. Install with `pip install "VIGOR[io,progress,plot] @ git+https://github.com/sanchestm/VIGOR"` (see the [README](../../../README.md)). Outputs are cleared: this notebook was checked against the 2.12.1 API but not re-executed for this release.

# Persisted project results and prediction

VIGOR 2.12.1. Run project notebooks 00–04 first. All earlier inputs retain their sample and trait coordinates.


In [ ]:
from pathlib import Path
import sys, os, json, warnings
import numpy as np
import pandas as pd
import xarray as xr

# Find ROOT/vigor_run/project_config.py regardless of notebook cwd.
def _find_run(start=None):
    p=Path(os.environ.get('VIGOR_PROJECT_ROOT', os.environ.get('DOMINO_PROJECT_ROOT', Path.cwd()))) if start is None else Path(start)
    for q in [p,*p.parents]:
        cand=q/'vigor_run'/'project_config.py'
        if cand.exists(): return cand.parent
        if q.name=='vigor_run' and (q/'project_config.py').exists(): return q
    raise FileNotFoundError('Set VIGOR_PROJECT_ROOT or place vigor_run under the project root.')
RUN=_find_run(); sys.path.insert(0,str(RUN))
from project_config import *
import vigor
print('Project root:',ROOT)
print('VIGOR:',vigor.__version__)
assert tuple(map(int, vigor.__version__.split('.')[:3])) >= (2, 12, 1), vigor.__version__


In [ ]:
train=pd.read_csv(TRAIN_CSV,index_col=0);train.index=train.index.astype(str)
store=vigor.open_store(ANALYSIS_STORE,mode="a",lazy=False)
cov=store.open_dataset("covariance/GE_full",lazy=False)
G=np.asarray(cov.G,float);E=np.asarray(cov.E,float)
if G.ndim==3:G=G.sum(axis=0)


In [ ]:
K=vigor.open_grm(store,component="additive",materialize=True)
result=vigor.mvPredict(train,K,G,E,accuracy="exact",preconditioner="diag",solver_maxiter=1000,blas_threads=BLAS_THREADS)
prediction=pd.DataFrame(np.asarray(result["phenotype_prediction"].to_pandas()),index=train.index,columns=train.columns)
store.write_predictions(prediction,name="phenotype_prediction",group="prediction/end_to_end",mode="a")
store.consolidate()
print("Prediction:",prediction.shape,"CG residual:",result.attrs.get("cg_relative_residual"))
